# Tutorial notebook for LCA-Modeller

Welcome to the *LCA-Modeller* library tutorial! *LCA-Modeller* offers a streamlined interface designed to facilitate the creation of **parametric LCA models** with **prospective capabilities**. It builds on the open-source libraries [*lca-algebraic*](https://lca-algebraic.readthedocs.io/) and [*premise*](https://premise.readthedocs.io/), so having a basic understanding of these tools, while not essential, is recommended.

The core functionality of *LCA-Modeller* revolves around reading a user-provided configuration file that defines the LCA model. From this configuration, *LCA-Modeller* generates a parametric LCA model, which can be evaluated for any parameter values using *lca-algebraic*'s built-in functions.
If prospective scenarios are provided, *premise* is used to adapt the EcoInvent database to future conditions. The parametric LCA model then selects the appropriate prospective databases based on the user-specified evaluation year.

In [34]:
# Import libraries
from lca_modeller.io.configuration import LCAProblemConfigurator
import lca_algebraic as agb
import numpy as np
import lca_modeller
lca_modeller.set_verbosity("normal")  # or "quiet", "normal", "verbose"

# Add data folder to path
import sys
import os.path as pth
sys.path.insert(0, '../..')
DATA_FOLDER = './data'

## 1. A basic example

In [35]:
# Provide the configuration file where the ecoinvent database, system model and impact categories are defined
configuration_file = pth.join(DATA_FOLDER, 'basic_example.yaml')
project, model, methods = LCAProblemConfigurator(configuration_file).generate()

Ecoinvent database 'ecoinvent-3.10-cutoff' already available; skipping import.
Set reset_project: True in configuration file to rebuild the Brightway project.
Building foreground model...
Foreground model successfully created.


In [36]:
# Provide values for parameters
parameters = {
    'a': 1.0,
    'b': 0.7,
    'c': 1.2,
    'd': 10.0,
}

# Compute impacts. The first call takes some time (compilation of the LCA functions) but next ones are much faster.
df = agb.compute_impacts(
    model,
    methods,
    **parameters
)
df

,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
model,89.6781,4.08824e-06,0.000217808,3.95525


In [37]:
# Compute impacts for several values at once
parameters = {
    'a': [1.0, 1.2, 1.4],  # list of values
    'b': [0.7, 0.5, 0.2],  # all lists must be of the same size
    'c': 1.2,  # constant value
    'd': 10.0,
}
agb.compute_impacts(
    model,
    methods,
    **parameters
)

,,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
a,b,,,,
1,0.7,89.6781,4.08824e-06,0.000217808,3.95525
1.2,0.5,87.7572,4.00675e-06,0.000213212,3.86261
1.4,0.2,68.5128,3.12332e-06,0.000166465,3.02178


In [38]:
# Another example
parameters = {
    'a': np.linspace(1.0, 2.0, 10),
    'b': np.linspace(0.7, 0.0, 10),
    'c': 1.2,
    'd': 10.0,
}
agb.compute_impacts(
    model,
    methods,
    **parameters
)

,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
model,89.6781,4.08824e-06,0.000217808,3.95525
1,92.311,4.21413e-06,0.000224257,4.06375
2,93.6146,4.27782e-06,0.000227465,4.1157
3,93.5787,4.27905e-06,0.000227411,4.11038
4,92.1966,4.21765e-06,0.000224079,4.04732
5,89.4633,4.0935e-06,0.000217458,3.92618
6,85.3753,3.9065e-06,0.00020754,3.7467
7,79.9298,3.65658e-06,0.000194318,3.50868
8,73.1247,3.3437e-06,0.000177788,3.21199
9,64.9585,2.96782e-06,0.000157947,2.8565


## 2. Get impacts by phase or subsystem

In [39]:
# Provide the configuration file where the ecoinvent database, system model and impact categories are defined
configuration_file = pth.join(DATA_FOLDER, 'basic_example_split_impacts.yaml')
project, model, methods = LCAProblemConfigurator(configuration_file).generate()

Ecoinvent database 'ecoinvent-3.10-cutoff' already available; skipping import.
Set reset_project: True in configuration file to rebuild the Brightway project.
Building foreground model...
Foreground model successfully created.


In [44]:
# Provide values for parameters
parameters = {
    'a': 1.0,
    'b': 0.7,
    'c': 1.2,
    'd': 10.0,
}

# Compute impacts and provide results by subsystem 
agb.compute_impacts(
    model,
    methods,
    axis='subsystem',
    **parameters
)

# Note: you might get SymPy warnings if running with lca-algebraic version <= 1.4.1. This will be fixed is version 1.4.2.

,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
subsystem,,,,
_other_,75.3618,4.03215e-06,0.000191308,3.01327
*sum*,75.3618,4.03215e-06,0.000191308,3.01327


In [45]:
# Compute impacts and provide results by food type 
agb.compute_impacts(
    model,
    methods,
    axis='foodtype',
    **parameters
)

# Note: compute_impacts() with axis is not yet compatible with array values for parameters. 
# A future version of lca_algebraic will make it possible to use both axis and arrays (https://github.com/oie-mines-paristech/lca_algebraic/issues/52)

,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
foodtype,,,,
_other_,75.3618,4.03215e-06,0.000191308,3.01327
*sum*,75.3618,4.03215e-06,0.000191308,3.01327


## 3. Prospective LCA

Caution: depending on the IAM, the locations might be changed compared to the original ecoinvent database.

In [42]:
# Provide the configuration file where the ecoinvent database, system model and impact categories are defined
configuration_file = pth.join(DATA_FOLDER, 'prospective_lca.yaml')
project, model, methods = LCAProblemConfigurator(configuration_file).generate()

Ecoinvent database 'ecoinvent-3.10-cutoff' already available; skipping import.
Set reset_project: True in configuration file to rebuild the Brightway project.
Building foreground model...
Foreground model successfully created.


In [43]:
# List databases installed in the project: projected dbs have been added
import bw2data
bw2data.databases

Databases dictionary with 8 object(s):
	Foreground DB
	biosphere3
	biosphere3-proxy
	ecoinvent-3.10-cutoff
	ecoinvent_cutoff_3.10_remind_SSP2-NPi_2020
	ecoinvent_cutoff_3.10_remind_SSP2-NPi_2030
	ecoinvent_cutoff_3.10_remind_SSP2-NPi_2040
	lca_modeller_user_flows

In [23]:
# Running a project with premise will automatically add the parameter 'year' to the model to enable the user to select the year of evaluation.
parameters = {
    'year': [2020, 2025, 2030, 2035, 2040],  # if the year falls between two ecoinvent datasets (e.g. 2020 and 2030), the result will be a linear interpolation of the two
    'a': [1.0, 1.2, 1.4, 1.6, 1.8],
    'b': [0.7, 0.5, 0.2, 0.1, 0.0],
    'c': 1.2,
    'd': 10.0,
}
agb.compute_impacts(
    model,
    methods,
    **parameters
)

,,,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
year,a,b,,,,
2020,1,0.7,87.344,4.08628e-06,0.00021194,3.46586
2025,1.2,0.5,81.938,3.98792e-06,0.000199644,3.23382
2030,1.4,0.2,61.1649,3.09529e-06,0.000149758,2.40657
2035,1.6,0.1,56.4987,2.93932e-06,0.000140864,2.23963
2040,1.8,0,49.3652,2.6408e-06,0.000125452,1.97408


## 4. Advanced examples

### Reference a previously defined activity and modify existing activities

In [12]:
# Provide the configuration file where the ecoinvent database, system model and impact categories are defined
configuration_file = pth.join(DATA_FOLDER, 'advanced_example.yaml')
project, model, methods = LCAProblemConfigurator(configuration_file).generate()

Ecoinvent database 'ecoinvent-3.10-cutoff' already available; skipping import.
Set reset_project: True in configuration file to rebuild the Brightway project.
Building foreground model...
Foreground model successfully created.


In [13]:
# Running a project with premise will automatically add the parameter 'year' to the model to enable the user to select the year of evaluation.
parameters = {
    'year': [2020, 2025, 2030, 2035, 2040],  # if the year falls between two ecoinvent datasets (e.g. 2020 and 2030), the result will be a linear interpolation of the two
    'a': [1.0, 1.2, 1.4, 1.6, 1.8],
    'b': [0.7, 0.5, 0.2, 0.1, 0.0],
    'c': 1.2,
    'd': 10.0,
    'p': [0.3, 0.2, 0.15, 0.12, 0.08]
}
agb.compute_impacts(
    model,
    methods,
    **parameters
)

,,,,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
year,a,b,p,,,,
2020,1,0.7,0.3,428.178,2.019e-05,0.0010386,16.8557
2025,1.2,0.5,0.2,336.05,1.64559e-05,0.000818482,13.1715
2030,1.4,0.2,0.15,214.919,1.09422e-05,0.000525951,8.39136
2035,1.6,0.1,0.12,174.096,9.1032e-06,0.000433799,6.85225
2040,1.8,0,0.08,135.363,7.27379e-06,0.000343745,5.37508


In [ ]:
# Running a project with premise will automatically add the parameter 'year' to the model to enable the user to select the year of evaluation.
parameters = {
    'year': [2020, 2025, 2030, 2035, 2040],  # if the year falls between two ecoinvent datasets (e.g. 2020 and 2030), the result will be a linear interpolation of the two
    'a': [1.0, 1.2, 1.4, 1.6, 1.8],
    'b': [0.7, 0.5, 0.2, 0.1, 0.0],
    'c': 1.2,
    'd': 10.0,
    'p': [0.3, 0.2, 0.15, 0.12, 0.08]
}
agb.compute_impacts(
    model,
    methods,
    **parameters
)

## 5. Define your own LCIA methods

In [14]:
# Provide the configuration file where the ecoinvent database, system model and impact categories are defined
configuration_file = pth.join(DATA_FOLDER, 'custom_lcia_methods.yaml')
project, model, methods = LCAProblemConfigurator(configuration_file).generate()

Ecoinvent database 'ecoinvent-3.10-cutoff' already available; skipping import.
Set reset_project: True in configuration file to rebuild the Brightway project.
Creating 1 custom LCIA method(s)...
Creating custom LCIA method: ('Custom methods', 'climate change', 'GWP100 (updated)')
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 3104.59it/s]

09:34:52+0200 [info     ] Vacuuming database            



LCIA matching file written to: /Users/f.pollet/POSTDOC/5 - Tools/LCA-Modeller/lca_modeller/notebook/data/lcia_gwp100_custom_updated.xlsx


Added 1 new biosphere flows
1 methods
130 cfs
0 unlinked cfs
Wrote 1 LCIA methods with 130 characterization factors


Building foreground model...
Foreground model successfully created.


In [15]:
# The custom method is automatically added to the list
methods

[('ecoinvent-3.10',
  'ReCiPe 2016 v1.03, midpoint (H)',
  'climate change',
  'global warming potential (GWP100)'),
 ('Custom methods', 'climate change', 'GWP100 (updated)')]

In [16]:
# Compute impacts with new method
parameters = {
    'a': [1.0, 1.2, 1.4, 1.6, 1.8],
    'b': [0.7, 0.5, 0.2, 0.1, 0.0],
    'c': 1.2,
    'd': 10.0,
}

agb.compute_impacts(
    model,
    methods,
    **parameters
)

/Users/f.pollet/POSTDOC/5 - Tools/LCA-Modeller/.venv/lib/python3.11/site-packages/bw2calc/multi_lca.py:288: UserWarning: All values in characterization matrix for ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, midpoint (H)', 'climate change', 'global warming potential (GWP100)') are zero
  warnings.warn(f"All values in characterization matrix for {key} are zero")


,,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]",climate change - GWP100 (updated)[kg CO2-Eq]
a,b,,
1,0.7,89.6781,123.057
1.2,0.5,87.7572,121.124
1.4,0.2,68.5128,101.802
1.6,0.1,65.1766,98.4502
1.8,0,58.7151,91.9618


# 6. Declare metadata for parameters
For example, declare min/max values and distribution type for each parameter to run a Monte Carlo using lca-algebraic's capabilities

In [17]:
# Provide the configuration file where the ecoinvent database, system model and impact categories are defined
configuration_file = pth.join(DATA_FOLDER, 'metadata_example.yaml')
project, model, methods = LCAProblemConfigurator(configuration_file).generate()

Ecoinvent database 'ecoinvent-3.10-cutoff' already available; skipping import.
Set reset_project: True in configuration file to rebuild the Brightway project.
Building foreground model...
Foreground model successfully created.


In [18]:
agb.list_parameters()

group,name,label,default,min,max,std,distrib,unit,db
,a,a,2,1,3,,linear,m,Foreground DB
,b,b,2,1,3,,triangle,kg,Foreground DB
,c,c,0.1,0,1,,fixed,kg,Foreground DB
,d,d,13.2,0,1,,fixed,,Foreground DB


In [19]:
agb.compute_impacts(model, methods)

,"ReCiPe 2016 v1.03, midpoint (H) - climate change[kg CO2-Eq]","ReCiPe 2016 v1.03, endpoint (H) - total: ecosystem quality[species.yr]","ReCiPe 2016 v1.03, endpoint (H) - total: human health[DALYs]","ReCiPe 2016 v1.03, endpoint (H) - total: natural resources[USD 2013]"
model,324.081,1.49235e-05,0.000788116,14.0994


In [20]:
agb.incer_stochastic_dashboard(model, methods, n=1024)

Generating samples ...


[WARNING] Param 'c' is marked as FIXED, but passed in parameters : ignored
[WARNING] Param 'd' is marked as FIXED, but passed in parameters : ignored


Transforming samples ...
Processing Sobol indices ...
Processing sobol for ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, midpoint (H)', 'climate change', 'global warming potential (GWP100)')
Processing sobol for ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H)', 'total: ecosystem quality', 'ecosystem quality')
Processing sobol for ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H)', 'total: human health', 'human health')
Processing sobol for ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H)', 'total: natural resources', 'natural resources')


/Users/f.pollet/POSTDOC/5 - Tools/LCA-Modeller/.venv/lib/python3.11/site-packages/SALib/util/__init__.py:274: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  names = list(pd.unique(groups))
